# FAST-UAV - Quadcopter Continuous Design Optimization
*Based on 1_Multirotor_Design workflow*

이 노트북은 연속 설계변수(continuous design variables) 기반으로 쿼드콥터를 최적화하는 예제입니다.

- 모델/최적화 설정: `multirotor_mdo.yaml`
- 입력 소스 파일: `problem_inputs_quadcopter.xml` (프로펠러 수 4로 설정)

In [1]:
# Import required libraries
import os.path as pth
import shutil
import fastoad.api as oad
from fastuav.utils.postprocessing.analysis_and_plots import *
from IPython.display import IFrame, display, HTML

display(HTML("<style>.container { width:95% !important; }</style>"))

c:\Users\user\.conda\envs\FastUAV_edit\lib\site-packages\stdatm\__init__.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
# Paths
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_mdo.yaml")
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_quadcopter.xml")

print("Configuration:", CONFIGURATION_FILE)
print("Source input:", SOURCE_FILE)

Configuration: ./data\configurations\multirotor_mdo.yaml
Source input: ./data\source_files\problem_inputs_quadcopter.xml


## 1) Model structure and inputs

In [ ]:
# Build and display N2 diagram
N2_FILE = pth.join(WORK_FOLDER_PATH, "n2_quadcopter.html")
oad.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)
IFrame(src=N2_FILE, width="100%", height=500)

In [ ]:
# Generate editable temporary inputs from source file
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

필요하면 위 Variable Viewer에서 입력값(페이로드, 체공시간 등)을 수정한 뒤 아래 최적화를 실행하세요.

## 2) Continuous MDO run

In [6]:
optim_problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)

Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.3696310566851395
            Iterations: 9
            Function evaluations: 9
            Gradient evaluations: 9
Optimization Complete
-----------------------------------


In [7]:
# Save output with a dedicated quadcopter filename
OUTPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_outputs.xml")
QUADCOPTER_OUTPUT_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_outputs_quadcopter_mdo.xml")
shutil.copy(OUTPUT_FILE, QUADCOPTER_OUTPUT_FILE)
print("Saved:", QUADCOPTER_OUTPUT_FILE)

Saved: ./data\source_files\problem_outputs_quadcopter_mdo.xml


In [ ]:
# Quick optimization summary
oad.optimization_viewer(CONFIGURATION_FILE)

In [ ]:
# Inspect all outputs
oad.variable_viewer(OUTPUT_FILE)

## 3) Plots

In [ ]:
# Geometry plot for optimized quadcopter
fig = multirotor_geometry_plot(QUADCOPTER_OUTPUT_FILE, name="Quadcopter optimized")
fig.show()

In [ ]:
# Mass breakdown plots
fig = mass_breakdown_sun_plot_drone(QUADCOPTER_OUTPUT_FILE)
fig.show()

fig = mass_breakdown_bar_plot_drone(QUADCOPTER_OUTPUT_FILE, name="Quadcopter optimized")
fig.show()

## 4) DJI 기준해와 Quadcopter 결과 비교

아래 셀은 `problem_outputs_DJI_M600_mdo.xml`(기준해)와 `problem_outputs_quadcopter_mdo.xml`(이 노트북 결과)를 비교합니다.

In [8]:
# Compare baseline DJI design and the new quadcopter design
OUTPUT_FILE_BASELINE = pth.join(SOURCE_FOLDER_PATH, "problem_outputs_DJI_M600_mdo.xml")
OUTPUT_FILE_QUAD = pth.join(SOURCE_FOLDER_PATH, "problem_outputs_quadcopter_mdo.xml")

if not pth.exists(OUTPUT_FILE_BASELINE):
    print("Baseline output file not found:", OUTPUT_FILE_BASELINE)
elif not pth.exists(OUTPUT_FILE_QUAD):
    print("Quadcopter output file not found:", OUTPUT_FILE_QUAD)
    print("Run the optimization section first (cells 10 to 11), then run this cell again.")
else:
    fig = multirotor_geometry_plot(OUTPUT_FILE_BASELINE, name="DJI M600 baseline")
    fig = multirotor_geometry_plot(OUTPUT_FILE_QUAD, name="Quadcopter optimized", fig=fig)
    fig.show()

    fig = mass_breakdown_bar_plot_drone(OUTPUT_FILE_BASELINE, name="DJI M600 baseline")
    fig = mass_breakdown_bar_plot_drone(OUTPUT_FILE_QUAD, name="Quadcopter optimized", fig=fig)
    fig.show()